In [ ]:
import re
import string
import time
import joblib
import pickle

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

In [ ]:
df = pd.read_csv('../data-collection/data.csv')

In [ ]:
df

In [ ]:
with open('./data/ukrainian-stopwords', 'r') as file:
    lines = file.readlines()

stopwords = []

for line in lines:
    words = line.split()
    stopwords.extend(words)

In [ ]:
# stopwords

## Machine Learning algorithms

### Text preprocessor transformer

In [ ]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        preprocessed_texts = []
        for text in X:
            preprocessed_text = self.preprocess_text(text)
            preprocessed_texts.append(preprocessed_text)
        return preprocessed_texts
    
    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub('\[.*?\]', '', text)
        text = re.sub("\\W"," ",text) 
        text = re.sub('https?://\S+|www\.\S+', '', text)
        text = re.sub('<.*?>+', '', text)
        text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
        text = re.sub('\n', '', text)
        text = re.sub('\w*\d\w*', '', text)
        return text

### Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=0)

In [ ]:
preprocessor = TextPreprocessor()

X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [ ]:
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### Logistic Regression model

In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_vec, y_train)

In [ ]:
pred_lr = model_lr.predict(X_test_vec)
print(classification_report(y_test, pred_lr))

### K-nearest Neighbors classifier

In [ ]:
model_kn = KNeighborsClassifier()
model_kn.fit(X_train_vec, y_train)

In [ ]:
pred_kn = model_kn.predict(X_test_vec)
print(classification_report(y_test, pred_kn))

### Gradient Boosting classifier

In [ ]:
model_gb = GradientBoostingClassifier(random_state=0)
model_gb.fit(X_train_vec, y_train)

In [ ]:
pred_gb = model_gb.predict(X_test_vec)
print(classification_report(y_test, pred_gb))

### Support Vector classifier

In [ ]:
model_sv = SVC(random_state=0)
model_sv.fit(X_train_vec, y_train)

In [ ]:
pred_sv = model_sv.predict(X_test_vec)
print(classification_report(y_test, pred_sv))

### Decision Tree classifier

In [ ]:
model_dt = DecisionTreeClassifier(random_state=0)
model_dt.fit(X_train_vec, y_train)

In [ ]:
pred_dt = model_dt.predict(X_test_vec)
print(classification_report(y_test, pred_dt))

### Random Forest classifier

In [ ]:
model_rf = RandomForestClassifier(random_state=0)
model_rf.fit(X_train_vec, y_train)

In [ ]:
pred_rf = model_rf.predict(X_test_vec)
print(classification_report(y_test, pred_rf))

### Stacking classifier (using 6 previous models)

In [ ]:
estimators = [
    ('lr', model_lr), 
    ('kn', model_kn),
    ('gb', model_gb),
    ('sv', model_sv),
    ('dt', model_dt), 
    ('rf', model_rf),
]

model_st = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
model_st.fit(X_train_vec, y_train)

In [ ]:
pred_st = model_st.predict(X_test_vec)
print(classification_report(y_test, pred_st))

## Pipeline for the final Stacking Classifier model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=0)

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text) 
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

In [ ]:
model_lr = LogisticRegression()
model_kn = KNeighborsClassifier()
model_gb = GradientBoostingClassifier(random_state=0)
model_sv = SVC(random_state=0)
model_dt = DecisionTreeClassifier(random_state=0)
model_rf = RandomForestClassifier(random_state=0)

In [ ]:
estimators = [
    ('lr', model_lr), 
    ('kn', model_kn),
    ('gb', model_gb),
    ('sv', model_sv),
    ('dt', model_dt), 
    ('rf', model_rf),
]

classifier = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

In [ ]:
vectorizer = TfidfVectorizer()

pipeline = Pipeline([
    ('tfidf-vectorizer', vectorizer),
    ('stacking-classifier', classifier)
])

pipeline.fit(X_train, y_train)

In [ ]:
predictions = pipeline.predict(X_test)
print(classification_report(y_test, predictions))

In [ ]:
pipeline.score(X_test, y_test)

In [ ]:
param = int(time.time())
joblib.dump(pipeline, f'../models/model_pipeline_{param}.pkl')

## Deep Learning algorithms

### Global params, helper functions and data preprocessing

In [ ]:
vocab_size = 10000
oov_tok = '<OOV>'
trunc_type = 'post'
padding_type = 'post'
max_length = 64

embedding_dim = 32
filters = 128
kernel_size = 10
gru_units = 64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=0)

In [ ]:
# tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
# tokenizer.fit_on_texts(X_train)

# with open('./data/tokenizer.pickle', 'wb') as handle:
#     pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open('./data/tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text) 
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
def preprocess(dataset, tokenizer):
    data = tokenizer.texts_to_sequences(dataset)
    data = pad_sequences(data, padding=padding_type, truncating=trunc_type, maxlen=max_length)
    return data

In [ ]:
def save_model(model, name):
    param = int(time.time())
    model.save(f'./models/model_{name}_{param}.keras')

In [ ]:
X_train.apply(preprocess_text)
X_test.apply(preprocess_text)

X_train_pad = preprocess(X_train, tokenizer)
X_test_pad = preprocess(X_test, tokenizer)

### LSTM

In [ ]:
model_lstm = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32,  return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_lstm.summary()

In [ ]:
model_lstm.compile(loss=tf.keras.losses.BinaryCrossentropy(),
                   optimizer=tf.keras.optimizers.Adam(1e-3),
                   metrics=['accuracy'])

history_lstm = model_lstm.fit(X_train_pad, 
                              y_train, 
                              epochs=10,
                              batch_size=256, 
                              validation_data=(X_test_pad, y_test),
                              callbacks=[early_stop])

In [ ]:
model_lstm.evaluate(X_test_pad, y_test)

In [ ]:
# save_model(model_lstm, 'lstm')

### CNN

In [ ]:
model_cnn = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Conv1D(filters, kernel_size, activation='relu'),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_cnn.summary()

In [ ]:
model_cnn.compile(loss=tf.keras.losses.BinaryCrossentropy(),
                  optimizer=tf.keras.optimizers.Adam(1e-3), 
                  metrics=['accuracy'])

history_cnn = model_cnn.fit(X_train_pad, 
                            y_train, 
                            epochs=10, 
                            batch_size=256, 
                            validation_data=(X_test_pad, y_test),
                            callbacks=[early_stop])

In [ ]:
model_cnn.evaluate(X_test_pad, y_test)

In [ ]:
# save_model(model_cnn, 'cnn')

### GRU

In [ ]:
model_gru = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GRU(gru_units),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_gru.summary()

In [ ]:
model_gru.compile(loss=tf.keras.losses.BinaryCrossentropy(),
                  optimizer=tf.keras.optimizers.Adam(1e-3), 
                  metrics=['accuracy'])

history_gru = model_gru.fit(X_train_pad, 
                            y_train, 
                            epochs=10, 
                            batch_size=256, 
                            validation_data=(X_test_pad, y_test),
                            callbacks=[early_stop])

In [ ]:
model_gru.evaluate(X_test_pad, y_test)

In [ ]:
# save_model(model_gru, 'gru')

## Ensembling 3 deep learning models and Stacking Classifier

In [ ]:
# model_lstm = tf.keras.models.load_model('./models/model_lstm_1714855304.keras')
# model_cnn = tf.keras.models.load_model('./models/model_cnn_1714853581.keras')
# model_gru = tf.keras.models.load_model('./models/model_gru_1714855993.keras')

model_st = joblib.load('./models/model_pipeline_1714870823.pkl')

In [ ]:
X_lstm = model_lstm.predict(X_train_pad)
X_cnn = model_cnn.predict(X_train_pad)
X_gru = model_gru.predict(X_train_pad)

X_st = model_st.predict_proba(X_train)
X_st = X_st[:, 1:]

In [ ]:
X_train_stacking = np.concatenate([X_lstm, X_cnn, X_gru, X_st], axis=-1)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(loss='binary_crossentropy',
              optimizer='adam', 
              metrics=['accuracy'])

history = model.fit(X_train_stacking, 
                    y_train, 
                    epochs=10, 
                    batch_size=32, 
                    validation_split=0.1)

In [ ]:
lstm_preds = model_lstm.predict(X_test_pad)
cnn_preds = model_cnn.predict(X_test_pad)
gru_preds = model_gru.predict(X_test_pad)

st_preds = model_st.predict_proba(X_test)
st_preds = st_preds[:, 1:]

In [ ]:
X_test_stacking = np.concatenate([lstm_preds, cnn_preds, gru_preds, st_preds], axis=-1)

In [ ]:
model.evaluate(X_test_stacking, y_test)

In [ ]:
model_st.score(X_test, y_test)